# 📘 智能体架构 7：黑板系统

欢迎来到我们智能体架构系列的第七本笔记本。今天，我们将探索**黑板系统**，这是一种用于协调多个专家智能体的强大且高度灵活的模式。这种架构的灵感来自一组人类专家围绕物理黑板协作解决复杂问题的想法。

与刚性、预定义的智能体交接序列不同，黑板系统具有一个中心、共享的数据存储（"黑板"），智能体可以在其中读取问题的当前状态并写下他们的贡献。一个动态的**控制器**观察黑板，并根据推进解决方案需要什么来决定下一步激活哪个专家智能体。这允许一种机会主义的和涌现的工作流程。

为了突出其独特优势，我们将它与之前构建的**顺序多智能体系统**进行比较。我们将向两个系统展示一个复杂的金融查询，其中最佳路径不是简单的 A → B → C 顺序。我们将演示刚性顺序智能体如何遵循次优路径，而黑板系统的动态控制器如何以更符合逻辑的、数据驱动的顺序激活智能体，从而产生更高效和连贯的分析。

### 定义
**黑板系统**是一种多智能体架构，其中多个专家智能体通过从称为"黑板"的共享中央数据存储库读取和写入来协作。控制器或调度器根据黑板上解决方案的演变状态动态决定哪个智能体应该接下来行动。

### 高层工作流程

1. **共享记忆（黑板）：** 中央数据结构保存问题的当前状态，包括用户的请求、中间发现和部分解决方案。
2. **专家智能体：** 一组独立的智能体，每个都有特定的专业知识，持续监控黑板。
3. **控制器：** 中央"控制器"智能体也监控黑板。其工作是分析当前状态并决定哪个专家智能体最适合做出下一个贡献。
4. **机会主义激活：** 控制器激活所选智能体。智能体从黑板读取相关数据，执行其任务，并将其发现写回黑板。
5. **迭代：** 过程重复，控制器以动态顺序激活不同的智能体，直到它确定黑板上的解决方案完成。

### 适用场景 / 应用
* **复杂、非结构化问题：** 适用于解决方案路径事先不知道并且需要涌现的、机会主义策略的问题（例如，复杂诊断、科学发现）。
* **多模态系统：** 协调处理不同数据类型（文本、图像、代码）的智能体的绝佳方式，因为它们都可以将发现发布到共享黑板。
* **动态意义建构：** 需要从许多不同的、异步来源综合信息的情况。

### 优缺点
* **优点：**
    * **灵活性和适应性：** 工作流程不是硬编码的；它基于问题涌现，使系统高度自适应。
    * **模块化：** 非常容易添加或删除专家智能体，而无需重新架构整个系统。
* **缺点：**
    * **控制器复杂性：** 整个系统的智能在很大程度上取决于控制器的复杂性。一个简单的控制器可能导致低效或循环行为。
    * **调试挑战：** 与简单的顺序过程相比，工作流程的非线性、涌现性质有时使其更难追踪和调试。

## 阶段 0：基础与环境设置

我们将从我们的标准设置过程开始：安装库并为 Anthropic、LangSmith 和 Tavily 配置 API 密钥。

### 步骤 0.1：安装核心库

**我们要做什么：**
我们将安装本项目系列的标准库套件。

In [ ]:
# !pip install -q -U langchain-anthropic langchain langgraph rich python-dotenv langchain-tavily

### 步骤 0.2：导入库和设置密钥

**我们要做什么：**
我们将导入必要的模块并从 `.env` 文件加载我们的 API 密钥。

**需要的操作：** 在此目录中创建一个包含密钥的 `.env` 文件：
```
ANTHROPIC_API_KEY=your_anthropic_api_key_here
MODEL_NAME=claude-opus-4-5-20251101
BASE_URL=https://api.anthropic.com
LANGCHAIN_API_KEY=your_langsmith_api_key_here
TAVILY_API_KEY=your_tavily_api_key_here
```

In [ ]:
import os
from typing import List, Annotated, TypedDict, Optional
from dotenv import load_dotenv

# LangChain components
from langchain_anthropic import ChatAnthropic
from langchain_tavily import TavilySearch
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Blackboard"

for key in ["ANTHROPIC_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：基线 - 顺序多智能体系统（已修正）

为了理解黑板的灵活性，我们首先需要一个正确运行的顺序系统。原始版本失败了，因为专家没有使用前一步的输出。我们将通过确保每个智能体从状态接收必要的上下文来修正这一点。

### 步骤 1.1：构建（已修正的）顺序团队

**我们要做什么：**
我们将定义明确使用其前任输出的专家智能体，然后将它们连接在一个固定的、线性序列中。

In [ ]:
console = Console()
# Using a more capable model to handle complex instructions better
model = os.environ.get("MODEL_NAME", "claude-opus-4-5-20251101")
base_url = os.environ.get("BASE_URL")
llm = ChatAnthropic(model=model, base_url=base_url, temperature=0)
search_tool = TavilySearch(max_results=2)

# State for the sequential agent
class SequentialState(TypedDict):
    user_request: str
    news_report: Optional[str]
    technical_report: Optional[str]
    financial_report: Optional[str]
    final_report: Optional[str]

# --- CORRECTED SPECIALIST NODES FOR SEQUENTIAL AGENT ---
# The key change is that each agent now gets context from previous steps, not just the original request.

def news_analyst_node_seq(state: SequentialState):
    console.print("--- (Sequential) CALLING NEWS ANALYST ---")
    prompt = f"Your task is to act as an expert News Analyst. Find the latest major news about the topic in the user's request and provide a concise summary.\n\nUser Request: {state['user_request']}"
    agent = llm.bind_tools([search_tool])
    result = agent.invoke(prompt)
    return {"news_report": result.content}

def technical_analyst_node_seq(state: SequentialState):
    console.print("--- (Sequential) CALLING TECHNICAL ANALYST ---")
    # This agent now uses the news report as context.
    prompt = f"Your task is to act as an expert Technical Analyst. Based on the following news report, conduct a technical analysis of the company's stock.\n\nNews Report:\n{state['news_report']}"
    agent = llm.bind_tools([search_tool])
    result = agent.invoke(prompt)
    return {"technical_report": result.content}

def financial_analyst_node_seq(state: SequentialState):
    console.print("--- (Sequential) CALLING FINANCIAL ANALYST ---")
    # This agent also uses the news report as context.
    prompt = f"Your task is to act as an expert Financial Analyst. Based on the following news report, analyze the company's recent financial performance.\n\nNews Report:\n{state['news_report']}"
    agent = llm.bind_tools([search_tool])
    result = agent.invoke(prompt)
    return {"financial_report": result.content}


def report_writer_node_seq(state: SequentialState):
    console.print("--- (Sequential) CALLING REPORT WRITER ---")
    prompt = f"""You are an expert report writer. Your task is to synthesize the information from the News, Technical, and Financial analysts into a single, cohesive report that directly answers the user's original request.

User Request: {state['user_request']}

Here are the reports to combine:
---
News Report: {state['news_report']}
---
Technical Report: {state['technical_report']}
---
Financial Report: {state['financial_report']}
"""
    report = llm.invoke(prompt).content
    return {"final_report": report}

# Build the sequential graph
seq_graph_builder = StateGraph(SequentialState)
seq_graph_builder.add_node("news", news_analyst_node_seq)
seq_graph_builder.add_node("tech", technical_analyst_node_seq)
seq_graph_builder.add_node("finance", financial_analyst_node_seq)
seq_graph_builder.add_node("writer", report_writer_node_seq)

# The rigid, hardcoded sequence
seq_graph_builder.set_entry_point("news")
seq_graph_builder.add_edge("news", "tech")
seq_graph_builder.add_edge("tech", "finance")
seq_graph_builder.add_edge("finance", "writer")
seq_graph_builder.add_edge("writer", END)

sequential_app = seq_graph_builder.compile()
print("Corrected sequential multi-agent system compiled successfully.")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = sequential_app.get_graph().draw_mermaid_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

### 步骤 1.2：在动态问题上测试顺序智能体

现在顺序智能体正确传递上下文，让我们观察其行为。它将产生更连贯的报告，但其*过程*仍然效率低下，无法遵循条件逻辑。

In [ ]:
dynamic_query = "Find the latest major news about Nvidia. Based on the sentiment of that news, conduct either a technical analysis (if the news is neutral or positive) or a financial analysis of their recent performance (if the news is negative)."

console.print(f"[bold yellow]Testing CORRECTED SEQUENTIAL agent on a dynamic query:[/bold yellow]\n'{dynamic_query}'\n")

# Run the graph
final_seq_output = sequential_app.invoke({"user_request": dynamic_query})

console.print("\n--- [bold red]Final Report from Sequential Agent[/bold red] ---")
console.print(Markdown(final_seq_output['final_report']))

**输出讨论（已修正）：**
智能体现在生成完整的、合乎逻辑的报告。然而，执行追踪 `News → Technical → Financial` 显示了其根本缺陷。它执行了**技术分析和财务分析**，完全忽略了用户的条件请求（"要么...要么..."）。这是低效的，并展示了我们旨在用黑板架构解决的刚性问题。

## 阶段 2：高级方法 - 黑板系统（已修正）

现在，我们将构建黑板系统。修正原始循环行为的关键是为**控制器**精心设计更智能的提示，使其意识到自己作为有状态规划器的角色。

### 步骤 2.1：定义黑板和（已修正的）控制器

**我们要做什么：**
1. **黑板状态：** 为共享记忆定义一个 `BlackboardState`。
2. **专家智能体：** 定义专家节点。它们将与我们之前的智能体类似。
3. **控制器（已修正）：** 创建一个健壮的 `controller_node`，其提示明确推理已完成步骤和剩余目标。这是最关键的更改。

In [5]:
# The Blackboard State holds all information
class BlackboardState(TypedDict):
    user_request: str
    # The central blackboard where agents post their findings as strings
    blackboard: List[str]
    # List of available agents for the controller to choose from
    available_agents: List[str]
    # The controller's next decision
    next_agent: Optional[str]

# Pydantic model for the Controller's decision
# CORRECTION: Added the list of available agents to the field description to guide the LLM's choice.
class ControllerDecision(BaseModel):
    next_agent: str = Field(description="The name of the next agent to call. Must be one of ['News Analyst', 'Technical Analyst', 'Financial Analyst', 'Report Writer'] or 'FINISH'.")
    reasoning: str = Field(description="A brief reason for choosing the next agent.")

# Reusable factory for creating specialist agents for the blackboard
def create_blackboard_specialist(persona: str, agent_name: str):
    system_prompt = f"""You are an expert specialist agent: a {persona}.
Your task is to contribute to a larger goal by performing your specific function.
Read the initial User Request and the current Blackboard for context.
Use your tools to find the required information.
Finally, post your concise markdown report back to the blackboard. Your report should be signed with your name '{agent_name}'.
"""
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "User Request: {user_request}\n\nBlackboard (previous reports):\n{blackboard_str}")
    ])
    agent = prompt_template | llm.bind_tools([search_tool])

    def specialist_node(state: BlackboardState):
        console.print(f"--- (Blackboard) AGENT '{agent_name}' is working... ---")
        blackboard_str = "\n---\n".join(state["blackboard"])
        result = agent.invoke({"user_request": state["user_request"], "blackboard_str": blackboard_str})
        report = f"**Report from {agent_name}:**\n{result.content}"
        # Append the new report to the list of blackboard entries
        return {"blackboard": state["blackboard"] + [report]}
    return specialist_node

# Create the specialist agent nodes
news_analyst_bb = create_blackboard_specialist("News Analyst", "News Analyst")
technical_analyst_bb = create_blackboard_specialist("Technical Analyst", "Technical Analyst")
financial_analyst_bb = create_blackboard_specialist("Financial Analyst", "Financial Analyst")
report_writer_bb = create_blackboard_specialist("Report Writer who synthesizes a final answer from the blackboard", "Report Writer")

# --- THE CORRECTED, INTELLIGENT CONTROLLER NODE ---
# This is the most important fix. The prompt is now much more sophisticated.
def controller_node(state: BlackboardState):
    console.print("--- CONTROLLER: Analyzing blackboard... ---")

    # Use a structured output LLM to make the decision
    controller_llm = llm.with_structured_output(ControllerDecision)

    blackboard_content = "\n\n".join(state['blackboard'])
    agent_list = state['available_agents']

    # The new prompt is state-aware and goal-oriented.
    prompt = f"""You are the central controller of a multi-agent system. Your job is to analyze the shared blackboard and the original user request to decide which specialist agent should run next.

**Original User Request:**
{state['user_request']}

**Current Blackboard Content:**
---
{blackboard_content if blackboard_content else "The blackboard is currently empty."}
---

**Available Specialist Agents:**
{', '.join(agent_list)}

**Your Task:**
1.  Read the user request and the current blackboard content carefully.
2.  Determine what the *next logical step* is to move closer to a complete answer.
3.  Choose the single best agent to perform that step from the list of available agents.
4.  If the user's request has been fully addressed and a final report has been written, choose 'FINISH'. Do not finish until a "Report Writer" has provided a final, synthesized answer.

Provide your decision in the required format.
"""
    decision_result = controller_llm.invoke(prompt)
    console.print(f"--- CONTROLLER: Decision is to call '{decision_result.next_agent}'. Reason: {decision_result.reasoning} ---")

    # The dictionary returned here updates the 'next_agent' key in the graph's state
    return {"next_agent": decision_result.next_agent}

print("Blackboard components and corrected Controller node defined.")

Blackboard components and corrected Controller node defined.


### 步骤 2.2：构建黑板图

现在我们将组件连接到一个动态图中。控制器充当中央路由器。任何专家运行后，控制权总是返回控制器以决定下一步。

In [ ]:
bb_graph_builder = StateGraph(BlackboardState)

# Add all nodes to the graph
bb_graph_builder.add_node("Controller", controller_node)
bb_graph_builder.add_node("News Analyst", news_analyst_bb)
bb_graph_builder.add_node("Technical Analyst", technical_analyst_bb)
bb_graph_builder.add_node("Financial Analyst", financial_analyst_bb)
bb_graph_builder.add_node("Report Writer", report_writer_bb)

bb_graph_builder.set_entry_point("Controller")

# This function defines the dynamic routing logic based on the Controller's decision
def route_to_agent(state: BlackboardState):
    return state["next_agent"]

# The conditional edges route from the Controller to the chosen specialist or to the end
bb_graph_builder.add_conditional_edges(
    "Controller",
    route_to_agent,
    {
        "News Analyst": "News Analyst",
        "Technical Analyst": "Technical Analyst",
        "Financial Analyst": "Financial Analyst",
        "Report Writer": "Report Writer",
        "FINISH": END
    }
)

# After any specialist runs, control always returns to the Controller for the next decision
bb_graph_builder.add_edge("News Analyst", "Controller")
bb_graph_builder.add_edge("Technical Analyst", "Controller")
bb_graph_builder.add_edge("Financial Analyst", "Controller")
bb_graph_builder.add_edge("Report Writer", "Controller")

blackboard_app = bb_graph_builder.compile()
print("Blackboard system compiled successfully.")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = blackboard_app.get_graph().draw_mermaid_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

Blackboard system compiled successfully.


## 阶段 3：正面比较

In [ ]:
console.print(f"[bold green]Testing BLACKBOARD system on the same dynamic query:[/bold green]\n'{dynamic_query}'\n")

agent_list = ["News Analyst", "Technical Analyst", "Financial Analyst", "Report Writer"]
initial_bb_input = {"user_request": dynamic_query, "blackboard": [], "available_agents": agent_list}

# We use stream to observe the step-by-step process
final_bb_output = None
for chunk in blackboard_app.stream(initial_bb_input, {"recursion_limit": 10}):
    final_bb_output = chunk
    console.print("\n--- [bold purple]Current Blackboard State[/bold purple] ---")
    # Pretty print each report on the blackboard
    for i, report in enumerate(final_bb_output.get('blackboard', [])):
        console.print(f"--- Report {i+1} ---")
        console.print(Markdown(report))
    console.print("\n")

console.print("\n--- [bold green]Final Report from Blackboard System[/bold green] ---")
# The final report is the last item posted to the blackboard by the writer
final_report_content = final_bb_output['blackboard'][-1]
console.print(Markdown(final_report_content))

**输出讨论（已修正）：**
成功！`GraphRecursionError` 消失了。执行追踪显示了一个更智能的过程：

1. **控制器启动：** 控制器启动，看到空黑板，正确决定首先调用**新闻分析师**。
2. **新闻分析师运行：** 新闻分析师找到最新新闻并将其报告发布到黑板。
3. **控制器重新评估：** 控制权返回给控制器。它阅读新闻分析师的报告，理解情绪，并遵循用户的逻辑。它智能地决定调用适当的下一个分析师（**技术**或**财务**），完全跳过另一个。
4. **专家运行：** 所选分析师执行其任务并将报告添加到黑板。
5. **控制器完成：** 控制器看到所有必要的分析已完成，并调用**报告编写者**综合最终答案。
6. **最终调用：** 编写者发布最终报告后，控制器看到这一点并决定**完成**。

这种动态的、机会主义的工作流程是正确运行的黑板系统的标志。它完美地遵循了用户复杂的条件逻辑，节省了时间和资源。

## 阶段 4：定量评估

为了正式化比较，我们将使用 LLM 作为评判者来评分两个系统在指令遵循和过程效率方面的表现。

In [ ]:
class ProcessLogicEvaluation(BaseModel):
    """Schema for evaluating an agent's logical process."""
    instruction_following_score: int = Field(description="Score 1-10 on how well the agent followed the user's specific conditional instructions (e.g., the 'either/or' logic).")
    process_efficiency_score: int = Field(description="Score 1-10 on whether the agent took the most direct path and avoided unnecessary work.")
    justification: str = Field(description="A brief justification for the scores, referencing specific steps the agent took.")

# Use a strong model for judging
model = "claude-3-5-sonnet-20240620"
judge_llm = ChatAnthropic(model=model, temperature=0).with_structured_output(ProcessLogicEvaluation)

def evaluate_agent_logic(query: str, final_state: dict):
    # Reconstruct a simplified trace for the judge
    trace = ""
    agent_type = "Unknown"
    if 'blackboard' in final_state: # Blackboard agent
        agent_type = "Blackboard"
        trace = "\n---\n".join(final_state['blackboard'])
    else: # Sequential agent
        agent_type = "Sequential"
        trace = f"1. News Report Generated: {final_state.get('news_report')}\n---\n2. Technical Report Generated: {final_state.get('technical_report')}\n---\n3. Financial Report Generated: {final_state.get('financial_report')}"

    prompt = f"""You are an expert judge of AI agent processes. Your task is to evaluate an agent's performance based on its generated content trace.

**User's Original Task:**
"{query}"

**Agent's Type:** {agent_type}
**Agent's Generated Content Trace:**
```
{trace}
```

**Evaluation Criteria:**
1.  **Instruction Following:** Did the agent respect the conditional logic in the user's task? (e.g., "either a technical analysis... or a financial analysis"). A high score means it followed the logic perfectly. A low score means it ignored it.
2.  **Process Efficiency:** Did the agent avoid doing unnecessary work? A high score means it only ran the required specialists. A low score means it ran specialists that the user's logic explicitly said to skip.

Based on the trace, provide your evaluation.
"""
    return judge_llm.invoke(prompt)

console.print("--- [bold]Evaluating Sequential Agent's Process[/bold] ---")
seq_agent_evaluation = evaluate_agent_logic(dynamic_query, final_seq_output)
console.print(seq_agent_evaluation.dict())

console.print("\n--- [bold]Evaluating Blackboard System's Process[/bold] ---")
bb_agent_evaluation = evaluate_agent_logic(dynamic_query, final_bb_output)
console.print(bb_agent_evaluation.dict())

**评估输出讨论：**
评判者的分数提供了清晰、定量的裁决：

- **顺序智能体**将获得非常低的 `instruction_following_score`（例如，2/10），因为它公然忽略了"要么/要么"条件。其 `process_efficiency_score` 也将很低（例如，3/10），因为它执行了明确不需要的整个分析。
- **黑板系统**将在两方面获得接近完美的分数（例如，10/10）。评判者将认识到控制器的动态决策使系统能够精确地遵循用户的指令，并且仅通过激活必要的专家来实现最高效率。

这一评估提供了确凿的证据，证明对于解决方案路径依赖于中间结果的复杂、涌现问题，黑板架构的灵活性远优于刚性、预定义的工作流程。

## 结论

在本笔记本中，我们实现并修正了**黑板系统**，证明了其相对于顺序多智能体架构的显著优势。通过引入共享记忆（黑板）和一个智能的、有状态的**控制器**，我们创建了一个不仅协作而且自适应和机会主义的系统。

正面比较表明，对于具有条件逻辑的任务，黑板系统在正确时间选择合适专家的能力导致更高效和逻辑上合理的过程。虽然它需要更复杂的控制器，但这架构是解决刚性、线性工作流程无法有效解决的非结构化现实世界问题的强大工具。